# 🚀 OMEGA vLLM Server — GPU Edition (Colab T4 Optimized)

**Optimized for:** Google Colab Free T4 (15GB VRAM) — 12hr max session
**Model:** Qwen/Qwen2-VL-7B-Instruct-AWQ (4-bit quantized)
**Features:**
- Rate-limited proxy to prevent abuse & Colab throttling
- Idle GPU auto-pause to save compute units
- Cloudflare tunnel with health monitoring
- Aggressive memory limits to stay under T4 ceiling

**⚠️ Colab Reality Check:**
- Free tier: ~12hr max session, ~15-30 GPU hrs/week
- No configuration guarantees 12hrs uninterrupted — Colab uses dynamic throttling
- These settings maximize your *probability* of reaching the limit

**Runtime:** `Runtime → Change runtime type → GPU` before running!

## Cell 1: GPU Check & Environment Setup

In [1]:
!nvidia-smi
# Expected: Tesla T4 with ~15109MiB VRAM
# If CPU only, go to Runtime → Change runtime type → GPU

# Downgrade transformers to vllm 0.6.4 compatible version
!pip install -q "transformers==4.46.3" "tokenizers>=0.19,<0.21"

from transformers.models.qwen2_vl.image_processing_qwen2_vl import make_batched_images
print("✅ make_batched_images OK")

import transformers, pathlib
fp8_path = pathlib.Path(transformers.__file__).parent / "integrations/finegrained_fp8.py"
print(f"fp8 file exists: {fp8_path.exists()}")   # should print False

Tue Jun 16 12:50:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2: Install vLLM & Launch Server (Conservative Settings)

In [2]:
!pip install -q vllm==0.6.4

import subprocess, time, urllib.request, pathlib, os

# Kill any existing vllm
subprocess.run(["pkill", "-f", "api_server"], capture_output=True)
time.sleep(2)

LOG = pathlib.Path("/tmp/vllm.log")
LOG.write_text("")

# ═══════════════════════════════════════════════════════════
# CONSERVATIVE FLAGS FOR COLAB T4 SURVIVAL
# ═══════════════════════════════════════════════════════════
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2-VL-7B-Instruct-AWQ",
        "--quantization", "awq",
        "--host", "0.0.0.0", "--port", "8000",
        "--dtype", "half",
        "--gpu-memory-utilization", "0.75",      # ← was 0.90 (saves ~2GB)
        "--max-model-len", "8192",              # ← was 16384 (halves KV cache)
        "--max-num-seqs", "4",                # ← ADD: cap concurrent requests
        "--max-num-batched-tokens", "8192",   # ← ADD: cap tokens per batch
        "--enforce-eager",
        "--trust-remote-code",
        "--limit-mm-per-prompt", "image=2",  # ← was 5
        "--disable-frontend-multiprocessing",
    ],
    stdout=open(LOG, "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,   # ← detaches from cell lifecycle
)

print(f"vLLM PID {vllm_proc.pid} — polling health...\n")

for i in range(90):
    time.sleep(10)
    lines = LOG.read_text().splitlines()
    if i % 3 == 0:
        print(f"t+{(i+1)*10}s:", lines[-1] if lines else "(silent)")
    if not pathlib.Path(f"/proc/{vllm_proc.pid}").exists():
        print("💀 Process died — log:")
        print("\n".join(lines[-30:]))
        break
    try:
        if urllib.request.urlopen("http://localhost:8000/health", timeout=2).status == 200:
            print(f"\n✅ vLLM healthy — PID {vllm_proc.pid} will survive cell exit")
            break
    except:
        pass
else:
    print(LOG.read_text()[-3000:])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.9/198.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.0/87.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.3 MB/s eta 0:00

## Cell 3: Rate-Limiting Proxy (Protects Against Public Abuse)

In [3]:
import subprocess, os

proxy_code = '''
import time, threading
from collections import defaultdict
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import StreamingResponse
import httpx, uvicorn

app = FastAPI()
VLLM_URL = "http://localhost:8000"

# Simple per-IP rate limiter: 10 req/min
request_log = defaultdict(list)
lock = threading.Lock()
RPM_LIMIT = 10

def check_rate(ip):
    now = time.time()
    with lock:
        request_log[ip] = [t for t in request_log[ip] if now - t < 60]
        if len(request_log[ip]) >= RPM_LIMIT:
            return False
        request_log[ip].append(now)
        return True

@app.api_route("/" + "{path:path}", methods=["GET","POST","PUT","DELETE"])
async def proxy(path: str, request: Request):
    ip = request.client.host
    if not check_rate(ip):
        raise HTTPException(429, "Rate limit: 10 req/min")
    body = await request.body()
    async with httpx.AsyncClient(timeout=300) as client:
        resp = await client.request(
            method=request.method,
            url=f"{VLLM_URL}/{path}",
            headers=dict(request.headers),
            content=body,
        )
    return StreamingResponse(
        iter([resp.content]),
        status_code=resp.status_code,
        headers=dict(resp.headers)
    )

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8001)
'''

with open("/tmp/proxy.py", "w") as f:
    f.write(proxy_code)

subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn", "httpx"], check=True)

subprocess.run(["pkill", "-f", "proxy.py"], capture_output=True)
time.sleep(1)

proxy_proc = subprocess.Popen(
    ["python", "/tmp/proxy.py"],
    stdout=open("/tmp/proxy.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"✅ Rate-limiting proxy PID {proxy_proc.pid} on :8001")

✅ Rate-limiting proxy PID 24019 on :8001


## Cell 4: Cloudflare Tunnel (Points to Proxy :8001)

In [4]:
import subprocess, re, time, os

if not os.path.exists("./cloudflared"):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "./cloudflared"
    ], check=True)
    os.chmod("./cloudflared", 0o755)

subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# Point tunnel to PROXY port 8001, NOT vLLM directly
cf = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8001"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    start_new_session=True,
)

for _ in range(30):
    line = cf.stdout.readline()
    if line: print(line.strip())
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        print(f"\n✅ URL:  {url}")
        print(f"   v1:   {url}/v1")
        print(f"\n   Verify: curl {url}/v1/models")
        break
    time.sleep(1)

2026-06-16T13:04:26Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-16T13:04:26Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-16T13:04:32Z INF +--------------------------------------------------------------------------------------------+
2026-06-16T13:04:32Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-16T13:04:32Z INF |  https://projector-acm-troy-islamic.trycloudflare.com 

## Cell 5: Verify End-to-End

In [7]:
import urllib.request, json, re, subprocess, time, socket
import ctypes

# ═══════════════════════════════════════════════════
# RE-EXTRACT URL from running cloudflared process
# ═══════════════════════════════════════════════════
print("🔍 Finding active Cloudflare tunnel...")

subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

import os
if not os.path.exists("./cloudflared"):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "./cloudflared"
    ], check=True)
    os.chmod("./cloudflared", 0o755)

cf = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8001"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    start_new_session=True,
)

url = None
for _ in range(60):
    line = cf.stdout.readline()
    if line:
        print(line.strip())
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        print(f"\n✅ URL captured: {url}")
        break
    time.sleep(1)

if not url:
    raise RuntimeError("❌ Could not extract Cloudflare URL")

# ═══════════════════════════════════════════════════
# FIX: Wait for DNS propagation + force resolver refresh
# ═══════════════════════════════════════════════════
print(f"\n⏳ Waiting for DNS propagation (trycloudflare needs 30-90s)...")

# Force glibc to re-read /etc/resolv.conf (clears negative DNS cache)
try:
    libc = ctypes.cdll.LoadLibrary('libc.so.6')
    libc.__res_init()
    print("🔄 DNS resolver cache flushed")
except Exception as e:
    print(f"⚠️ Could not flush DNS cache: {e}")

# Retry with exponential backoff
max_retries = 15
base_delay = 3
for attempt in range(1, max_retries + 1):
    # Re-flush DNS before each attempt
    try:
        libc.__res_init()
    except:
        pass

    try:
        print(f"\n🌐 Attempt {attempt}/{max_retries}: Testing {url}/v1/models")
        req = urllib.request.Request(
            f"{url}/v1/models",
            headers={"Content-Type": "application/json"}
        )
        resp = json.loads(urllib.request.urlopen(req, timeout=15).read())
        print("\n✅ SUCCESS!")
        print(json.dumps(resp, indent=2))
        break
    except urllib.error.URLError as e:
        print(f"   ❌ Failed: {e}")
        if attempt == max_retries:
            print("\n💀 All retries exhausted. Possible causes:")
            print("   - Cloudflare tunnel died (check /tmp/vllm.log)")
            print("   - Proxy on :8001 is not running")
            print("   - trycloudflare.com is temporarily down")
            raise
        delay = min(base_delay * (2 ** (attempt - 1)), 60)  # cap at 60s
        print(f"   ⏱️  Retrying in {delay}s...")
        time.sleep(delay)

🔍 Finding active Cloudflare tunnel...
2026-06-16T13:12:46Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-16T13:12:46Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-16T13:12:51Z INF +--------------------------------------------------------------------------------------------+
2026-06-16T13:12:51Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-16T13:12:51Z INF |  https://disco-k

## Cell 6: Keep Alive + Idle GPU Auto-Pause (RUN THIS LAST)

In [ ]:
import urllib.request, subprocess, pathlib, time

idle_since = None
IDLE_SHUTDOWN_MINUTES = 20

while True:
    time.sleep(60)
    now = time.strftime("%H:%M:%S")

    # Health check
    try:
        r = urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print(f"♥ Heartbeat {now} — Health: ✅ {r.status}")
    except Exception as e:
        print(f"♥ Heartbeat {now} — Health: ❌ {e}")

    # Idle detection via metrics
    try:
        import urllib.request
        metrics = urllib.request.urlopen("http://localhost:8000/metrics", timeout=5).read().decode()
        running_lines = [l for l in metrics.splitlines() if "vllm:num_requests_running" in l and not l.startswith("#")]
        if running_lines:
            running = int(running_lines[-1].split()[-1])
            if running == 0:
                if idle_since is None:
                    idle_since = time.time()
                    print(f"   💤 GPU idle detected...")
                elif (time.time() - idle_since) > IDLE_SHUTDOWN_MINUTES * 60:
                    print(f"   💤 Idle {IDLE_SHUTDOWN_MINUTES}min — pausing vLLM to save compute units")
                    subprocess.run(["pkill", "-STOP", "-f", "api_server"], capture_output=True)
                    print(f"   ⏸️  vLLM paused. Resume on next request.")
            else:
                if idle_since:
                    subprocess.run(["pkill", "-CONT", "-f", "api_server"], capture_output=True)
                    print(f"   ▶️  vLLM resumed — requests active")
                idle_since = None
    except Exception as e:
        pass

    # Show last log lines
    log = pathlib.Path("/tmp/vllm.log")
    if log.exists():
        lines = log.read_text().splitlines()
        print(f"\n--- Last 5 log lines ({len(lines)} total) ---")
        print("\n".join(lines[-5:]))

    print("═" * 50)

♥ Heartbeat 13:15:18 — Health: ✅ 200

--- Last 5 log lines (121 total) ---
INFO 06-16 13:15:00 metrics.py:449] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 0.0%, CPU KV cache usage: 0.0%.
INFO:     2401:4900:1c3b:1140:6909:bdf8:bca4:4631:0 - "GET /v1/models HTTP/1.1" 200 OK
INFO 06-16 13:15:10 metrics.py:449] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 0.0%, CPU KV cache usage: 0.0%.
INFO:     127.0.0.1:47358 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:47362 - "GET /metrics HTTP/1.1" 200 OK
══════════════════════════════════════════════════
♥ Heartbeat 13:16:18 — Health: ✅ 200

--- Last 5 log lines (129 total) ---
INFO 06-16 13:15:50 metrics.py:449] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Swapped: 0 reqs, Pending: 0 re